# Sparse Autoencoders: The Foundation

**Survey Reference:** Section 7.1 - Sparse Autoencoders: The Foundation

## Overview

**Sparse Autoencoders (SAEs)** are the primary tool for extracting interpretable features from neural network activations. They learn to decompose polysemantic representations into sparse combinations of monosemantic features.

### The Core Idea

Given activations $x \in \mathbb{R}^d$, an SAE learns:

1. **Encode:** $f = \sigma(W_e x + b_e)$ where $f \in \mathbb{R}^m$ is sparse, $m >> d$
2. **Decode:** $\hat{x} = W_d f + b_d$

## Learning Objectives

1. Build an SAE from scratch in PyTorch
2. Understand the reconstruction vs. sparsity tradeoff
3. Train an SAE on transformer activations
4. Interpret learned features

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. SAE Architecture

The standard SAE has:
- **Expansion factor:** Usually 4x to 64x (m = expansion * d)
- **ReLU activation:** Creates sparsity in the feature space
- **Tied or untied weights:** $W_d = W_e^T$ or learned separately

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    A basic Sparse Autoencoder for mechanistic interpretability.
    
    Architecture:
        input (d) -> encoder (d -> m) -> ReLU -> decoder (m -> d)
    """
    
    def __init__(self, d_model: int, n_features: int, tied_weights: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        self.tied_weights = tied_weights
        
        # Encoder
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder
        if tied_weights:
            self.W_dec = None  # Will use W_enc.T
        else:
            self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input to sparse feature activations."""
        # Center the input (optional but common)
        x_centered = x - self.b_dec
        # Linear projection + ReLU
        pre_acts = x_centered @ self.W_enc + self.b_enc
        return F.relu(pre_acts)
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode feature activations back to input space."""
        if self.tied_weights:
            W_d = self.W_enc.T
        else:
            W_d = self.W_dec
        return f @ W_d + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Full forward pass. Returns (reconstruction, feature_activations)."""
        f = self.encode(x)
        x_hat = self.decode(f)
        return x_hat, f
    
    @property
    def feature_directions(self) -> torch.Tensor:
        """Get normalized decoder directions for each feature."""
        if self.tied_weights:
            W = self.W_enc.T
        else:
            W = self.W_dec
        return W / W.norm(dim=1, keepdim=True)

## 2. Loss Function

The SAE loss has two components:

$$\mathcal{L} = \underbrace{\|x - \hat{x}\|_2^2}_{\text{reconstruction}} + \lambda \underbrace{\|f\|_1}_{\text{sparsity}}$$

- **Reconstruction loss:** Preserve information
- **L1 sparsity penalty:** Encourage sparse activations
- **λ (lambda):** Balances the two objectives

In [ ]:
def sae_loss(x: torch.Tensor, x_hat: torch.Tensor, f: torch.Tensor, 
             l1_coef: float = 1e-3) -> tuple[torch.Tensor, dict]:
    """
    Compute SAE loss with reconstruction and sparsity terms.
    
    Returns:
        total_loss: Combined loss for backprop
        metrics: Dict with individual loss components
    """
    # Reconstruction loss (MSE)
    recon_loss = (x - x_hat).pow(2).mean()
    
    # L1 sparsity loss
    l1_loss = f.abs().mean()
    
    # Total loss
    total_loss = recon_loss + l1_coef * l1_loss
    
    # Compute additional metrics
    with torch.no_grad():
        # Fraction of variance explained
        fvu = (x - x_hat).var() / x.var()
        fve = 1 - fvu
        
        # Average number of active features per sample
        l0 = (f > 0).float().sum(dim=1).mean()
    
    metrics = {
        'loss': total_loss.item(),
        'recon_loss': recon_loss.item(),
        'l1_loss': l1_loss.item(),
        'fve': fve.item(),
        'l0': l0.item(),
    }
    
    return total_loss, metrics

## 3. Training on Synthetic Data

First, let's verify our SAE works on synthetic data with known ground truth features.

In [ ]:
def generate_synthetic_data(n_samples: int, d_model: int, n_true_features: int, 
                            sparsity: float = 0.9) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Generate synthetic data from known sparse features.
    
    Returns:
        X: Data matrix [n_samples, d_model]
        F_true: True feature activations [n_samples, n_true_features]
        W_true: True feature directions [n_true_features, d_model]
    """
    # Random feature directions (normalized)
    W_true = torch.randn(n_true_features, d_model)
    W_true = W_true / W_true.norm(dim=1, keepdim=True)
    
    # Sparse feature activations
    F_true = torch.rand(n_samples, n_true_features) * (torch.rand(n_samples, n_true_features) > sparsity).float()
    
    # Generate data
    X = F_true @ W_true
    
    return X, F_true, W_true

# Generate data
d_model = 64
n_true_features = 128
n_samples = 10000

X, F_true, W_true = generate_synthetic_data(n_samples, d_model, n_true_features)
print(f"Data shape: {X.shape}")
print(f"Average L0: {(F_true > 0).float().sum(dim=1).mean():.1f}")

In [ ]:
def train_sae(sae: SparseAutoencoder, X: torch.Tensor, 
              n_epochs: int = 100, batch_size: int = 256, 
              lr: float = 1e-3, l1_coef: float = 1e-3) -> list[dict]:
    """
    Train an SAE on data.
    """
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)
    history = []
    
    for epoch in tqdm(range(n_epochs)):
        # Shuffle data
        perm = torch.randperm(X.size(0))
        epoch_metrics = []
        
        for i in range(0, X.size(0), batch_size):
            batch = X[perm[i:i+batch_size]].to(device)
            
            optimizer.zero_grad()
            x_hat, f = sae(batch)
            loss, metrics = sae_loss(batch, x_hat, f, l1_coef)
            loss.backward()
            optimizer.step()
            
            epoch_metrics.append(metrics)
        
        # Average metrics
        avg_metrics = {k: np.mean([m[k] for m in epoch_metrics]) for k in epoch_metrics[0]}
        history.append(avg_metrics)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss={avg_metrics['loss']:.4f}, FVE={avg_metrics['fve']:.4f}, L0={avg_metrics['l0']:.1f}")
    
    return history

# Train SAE
sae = SparseAutoencoder(d_model, n_features=256).to(device)
history = train_sae(sae, X, n_epochs=100, l1_coef=5e-3)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot([h['loss'] for h in history])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss')

axes[1].plot([h['fve'] for h in history])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('FVE')
axes[1].set_title('Fraction of Variance Explained')

axes[2].plot([h['l0'] for h in history])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('L0')
axes[2].set_title('Average Active Features')

plt.tight_layout()
plt.show()

## 4. Training on Real Model Activations

Now let's train an SAE on actual transformer activations.

In [ ]:
# TODO: Collect activations from a transformer model
# from transformers import AutoModelForCausalLM, AutoTokenizer

def collect_activations(model, tokenizer, texts, layer_idx):
    """Collect MLP activations from a specific layer."""
    pass

## 5. Interpreting Learned Features

After training, we analyze what each feature represents by:
1. Finding max-activating examples
2. Examining decoder directions
3. Testing feature interventions

In [ ]:
def find_max_activating_examples(sae, X, feature_idx, top_k=10):
    """Find examples that maximally activate a feature."""
    with torch.no_grad():
        _, f = sae(X.to(device))
        acts = f[:, feature_idx].cpu()
    
    top_indices = acts.argsort(descending=True)[:top_k]
    return top_indices, acts[top_indices]

# TODO: Analyze learned features

## Key Takeaways

1. **SAEs decompose activations** into sparse, interpretable features
2. **L1 penalty** controls sparsity (tradeoff with reconstruction)
3. **Expansion factor** (m/d) determines feature capacity
4. **Feature interpretation** requires finding max-activating examples
5. **Training is sensitive** to hyperparameters (L1 coefficient, learning rate)

## Next Steps

→ **02_transcoders.ipynb**: Learn features that map across MLP layers